In [0]:
#Conectando o notebook da trnsformação Silver a do Bronze
CATALOG = "mvp_mmgd"
SCHEMA_BRONZE = "bronze"
SCHEMA_SILVER = "silver"

df_empreendimento_bronze = spark.table(f"{CATALOG}.{SCHEMA_BRONZE}.empreendimento_gd")
df_tecnica_bronze = spark.table(f"{CATALOG}.{SCHEMA_BRONZE}.tecnica_fotovoltaica")

print(f"empreendimento_gd: {df_empreendimento_bronze.count():,} linhas")
print(f"tecnica_fotovoltaica: {df_tecnica_bronze.count():,} linhas")

In [0]:
from pyspark.sql.functions import regexp_replace, col
#Corrigindo a virgula decimal e convertendo para Double, necessário pois sem essa correção a soma e o filtro não entenderam como número e sim como texto.
df_empreendimento_silver = df_empreendimento_bronze.withColumn(
    "MdaPotenciaInstaladaKW",
    regexp_replace(col("MdaPotenciaInstaladaKW"), ",", ".").cast("double")
)

df_empreendimento_silver.select("MdaPotenciaInstaladaKW").printSchema()
display(df_empreendimento_silver.select("MdaPotenciaInstaladaKW").limit(10))

In [0]:
#Fazendo a correção das colunas numéricas da tabela técnica
colunas_numericas_tecnica = [
    "MdaAreaArranjo",
    "MdaPotenciaInstalada",
    "MdaPotenciaModulos",
    "MdaPotenciaInversores",
]

df_tecnica_silver = df_tecnica_bronze

for coluna in colunas_numericas_tecnica:
    df_tecnica_silver = df_tecnica_silver.withColumn(
        coluna,
        regexp_replace(col(coluna), ",", ".").cast("double")
    )

df_tecnica_silver.select(colunas_numericas_tecnica).printSchema()
display(df_tecnica_silver.select(colunas_numericas_tecnica).limit(10))

In [0]:
#Padronizando o nome dos fabricantes de módulos e inverosres, removendo os espaços em branco e convertendo o texto para maiúsculas
from pyspark.sql.functions import upper, trim

colunas_fabricante = ["NomFabricanteModulo", "NomFabricanteInversor"]

for coluna in colunas_fabricante:
    df_tecnica_silver = df_tecnica_silver.withColumn(
        coluna,
        trim(upper(col(coluna)))
    )

display(df_tecnica_silver.select(colunas_fabricante).distinct().limit(20))

In [0]:
from pyspark.sql.functions import when

#Aparecerem alguns problemas na analise: pontos outliers nas duas colunas que irão atrapalhar as analises, como NÃO HÁ, NÃO CONSTA
PLACEHOLDERS_NULO = ["NÃO HÁ", "NÃO CONSTA", ""]

# Nome padronizados para o mesmo fabricante
MAPA_SINONIMOS_FABRICANTE = {
    "CANADIAN": "CANADIAN SOLAR",
    "CANADIN SOLAR INC.": "CANADIAN SOLAR",
}

def padronizar_fabricante(coluna_nome):
    expressao = trim(upper(col(coluna_nome)))
    # Placeholders de texto viram nulo
    expressao = when(expressao.isin(PLACEHOLDERS_NULO), None).otherwise(expressao)
    # Aplica cada sinônimo conhecido
    for variante, nome_padrao in MAPA_SINONIMOS_FABRICANTE.items():
        expressao = when(expressao == variante, nome_padrao).otherwise(expressao)
    return expressao

for coluna in colunas_fabricante:
    df_tecnica_silver = df_tecnica_silver.withColumn(coluna, padronizar_fabricante(coluna))

display(df_tecnica_silver.select(colunas_fabricante).distinct().limit(20))

In [0]:
#Investigando valores fora do padrão em DatConexao que ainda permanecem como string

from pyspark.sql.functions import try_to_date

print("Valores problemáticos em DatConexao:")
display(
    df_tecnica_silver
    .filter(col("DatConexao").isNotNull() & try_to_date(col("DatConexao"), "yyyy-MM-dd").isNull())
    .select("DatConexao")
    .distinct()
    .limit(20)
)

In [0]:
#Descobrindo quantas células estão com problemas de formataçã na DatConexao
qtd_problematico_qtd = df_tecnica_silver.filter(
    col("QtdModulos").isNotNull() & expr("try_cast(QtdModulos AS INT)").isNull()
).count()

print(f"Registros com QtdModulos corrompida: {qtd_problematico_qtd:,} de {qtd_total_tecnica:,} ({qtd_problematico_qtd/qtd_total_tecnica:.3%})")

In [0]:
#Convertendo colunas QtdModulos e DatConexao para seus tipo corretos
from pyspark.sql.functions import try_to_date, expr

df_tecnica_silver = (
    df_tecnica_silver
    .withColumn("QtdModulos", expr("try_cast(QtdModulos AS INT)"))
    .withColumn("DatConexao", try_to_date(col("DatConexao"), "yyyy-MM-dd"))
)

df_tecnica_silver.select("QtdModulos", "DatConexao").printSchema()

In [0]:
#Por fim, é necessário verificar a quantidade de valores duplicados para garantir que cada empreendimento está aparecendo uma única vez, registros duplicados inflam os números artificialmente.


qtd_linhas = df_empreendimento_silver.count()
qtd_codigos_unicos = df_empreendimento_silver.select("CodEmpreendimento").distinct().count()

print(f"empreendimento_gd — linhas: {qtd_linhas:,} | códigos únicos: {qtd_codigos_unicos:,}")
if qtd_linhas == qtd_codigos_unicos:
    print("OK — nenhuma duplicata de CodEmpreendimento.")
else:
    print(f"ATENÇÃO — {qtd_linhas - qtd_codigos_unicos:,} linhas duplicadas encontradas.")

qtd_linhas_tec = df_tecnica_silver.count()
qtd_codigos_unicos_tec = df_tecnica_silver.select("CodGeracaoDistribuida").distinct().count()

print(f"\ntecnica_fotovoltaica — linhas: {qtd_linhas_tec:,} | códigos únicos: {qtd_codigos_unicos_tec:,}")
if qtd_linhas_tec == qtd_codigos_unicos_tec:
    print("OK — nenhuma duplicata de CodGeracaoDistribuida.")
else:
    print(f"ATENÇÃO — {qtd_linhas_tec - qtd_codigos_unicos_tec:,} linhas duplicadas encontradas.")

In [0]:
#Salvando as tabelas Silver como Delta

from pyspark.sql.functions import current_timestamp, lit

df_empreendimento_final = (
    df_empreendimento_silver
    .withColumn("_dth_transformacao", current_timestamp())
    .withColumn("_camada", lit("silver"))
)

df_tecnica_final = (
    df_tecnica_silver
    .withColumn("_dth_transformacao", current_timestamp())
    .withColumn("_camada", lit("silver"))
)

(
    df_empreendimento_final.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SCHEMA_SILVER}.empreendimento_gd")
)

(
    df_tecnica_final.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SCHEMA_SILVER}.tecnica_fotovoltaica")
)

print("Tabelas Silver salvas com sucesso.")

In [0]:
#Documentando no catálogo as tabelas Silver
spark.sql(f"""
    COMMENT ON TABLE {CATALOG}.{SCHEMA_SILVER}.empreendimento_gd IS
    'Cadastro de empreendimentos MMGD. Correções: vírgula decimal em MdaPotenciaInstaladaKW. Recorte: Região Sul. Camada: Silver.'
""")

spark.sql(f"""
    COMMENT ON TABLE {CATALOG}.{SCHEMA_SILVER}.tecnica_fotovoltaica IS
    'Informações técnicas fotovoltaicas. Correções: vírgula decimal, padronização de fabricante, DatConexao/QtdModulos convertidos. Recorte: Região Sul. Camada: Silver.'
""")

for tabela in ["empreendimento_gd", "tecnica_fotovoltaica"]:
    nome_completo = f"{CATALOG}.{SCHEMA_SILVER}.{tabela}"
    print(f"\n=== {nome_completo} ===")
    spark.table(nome_completo).printSchema()